# The `os` Module: Working Directory and Paths

**Session 06 (2026-10-24) · Priority 1 · Notebook 2 of 6**

Before you can read or write files, you need to know **where** they are. The `os` module answers questions like "which folder am I in?", "what's in this folder?" and "how do I build a path that works on every computer?".

What you'll learn:

1. What the `os` module is
2. The current working directory: `os.getcwd()`, `os.chdir()`
3. Absolute (full) paths vs relative paths
4. Building paths: `os.path.join()`
5. Inspecting paths: `exists`, `isfile`, `isdir`, `basename`, `dirname`, `splitext`
6. Listing a folder: `os.listdir()`
7. Creating, renaming and removing files and folders
8. Paths in `.py` scripts vs notebooks
9. Practice

---
## 1. What is `os`?

**What it is:** `os` is a built-in module (nothing to install) that lets Python talk to the **operating system**: folders, files, paths and environment variables. `os.path` is a sub-module just for working with paths.

**Behind the scenes:** each `os` function asks the operating system to do the actual work. The same Python code runs on macOS, Linux and Windows, because `os` uses the right method for each one.

Since this module is *Python & Linux*, here is how the `os` functions map to shell commands you already know:

| Python | Linux / macOS shell |
|---|---|
| `os.getcwd()` | `pwd` |
| `os.chdir(path)` | `cd path` |
| `os.listdir(path)` | `ls path` |
| `os.mkdir(path)` | `mkdir path` |
| `os.makedirs(path, exist_ok=True)` | `mkdir -p path` |
| `os.rename(src, dst)` | `mv src dst` |
| `os.remove(path)` | `rm path` |
| `os.rmdir(path)` | `rmdir path` |

In [ ]:
import os

print(os.name)     # 'posix' on macOS/Linux, 'nt' on Windows
print(os.sep)      # the path separator: '/' on macOS/Linux, '\' on Windows

---
## 2. The current working directory

**What it is:** every running program has a **current working directory** (**cwd**): the folder it's "standing in". Relative paths are looked up from here.

**Behind the scenes:** the cwd belongs to the **running process** (the notebook's kernel, or the `python` command in a terminal). It's set to wherever the program was **started from**, not to where the `.py` file is saved. In a notebook, it's usually the notebook's own folder.

### `os.getcwd()`

**Syntax:** `os.getcwd()` (no arguments)
**Returns:** the cwd as an absolute path string.

In [ ]:
cwd = os.getcwd()
print(cwd)
print(type(cwd))

### `os.chdir()`

**Syntax:** `os.chdir(path)`
**Returns:** `None`. It changes the cwd for the rest of the program.

Changing the cwd affects **every** relative path used afterwards, so always change it back:

In [ ]:
original = os.getcwd()

os.chdir("..")                      # move up one folder, like `cd ..`
print("now in :", os.getcwd())

os.chdir(original)                  # change back, or later relative paths break
print("back in:", os.getcwd())

---
## 3. Absolute vs relative paths

| | Absolute (full) path | Relative path |
|---|---|---|
| Starts from | the root of the file system (`/` on macOS/Linux, `C:\` on Windows) | the **current working directory** |
| Example | `/Users/asha/fde-bootcamp/data/notes.txt` | `data/notes.txt` |
| Means the same thing everywhere? | Yes | No: it depends on the cwd |
| Works on someone else's computer? | Rarely (usernames and folders differ) | Yes, if the folder layout is the same |

Two special names in relative paths:

- `.` means the current folder
- `..` means the parent folder (one level up)

**Behind the scenes:** a relative path is glued onto the cwd: `cwd + "/" + relative_path`. `os.path.abspath()` shows you the result.

**Syntax:** `os.path.abspath(path)`
**Returns:** the absolute path string. It only builds the string; the file doesn't have to exist.

In [ ]:
print(os.path.abspath("notes.txt"))       # cwd + notes.txt
print(os.path.abspath("../notes.txt"))    # parent folder + notes.txt
print(os.path.abspath("."))               # the cwd itself

`os.path.isabs(path)` returns `True` if a path is already absolute:

In [ ]:
print(os.path.isabs("/Users/asha/notes.txt"))   # True
print(os.path.isabs("notes.txt"))               # False

**Rule of thumb:** inside a project, use **relative** paths (so the project works on any machine), and build them with `os.path.join()`.

---
## 4. Building paths: `os.path.join()`

**What it is:** joins folder and file names into one path, using the correct separator for the operating system.

**Behind the scenes:** it only builds a **string**. It doesn't check whether the path exists and doesn't create anything.

**Syntax:** `os.path.join(part_1, part_2, ...)`
**Returns:** a single path string.

**Why not just write `"data" + "/" + "notes.txt"`?** That hard-codes `/`, which isn't the Windows separator, and it's easy to end up with a doubled or missing slash.

In [ ]:
path = os.path.join("data", "2026", "notes.txt")
print(path)                # data/2026/notes.txt on macOS/Linux

full_path = os.path.join(os.getcwd(), "data", "notes.txt")
print(full_path)           # an absolute path built from the cwd

**Gotcha:** if one of the later parts is an absolute path, everything before it is thrown away:

In [ ]:
print(os.path.join("data", "/tmp", "notes.txt"))   # '/tmp/notes.txt'

---
## 5. Inspecting paths

Some functions only work on the path **string**. Others ask the operating system about the **disk**.

| Function | Returns | Checks the disk? |
|---|---|---|
| `os.path.basename(p)` | the last part, e.g. `"notes.txt"` | No |
| `os.path.dirname(p)` | everything before the last part | No |
| `os.path.split(p)` | a tuple `(dirname, basename)` | No |
| `os.path.splitext(p)` | a tuple `(path_without_extension, ".ext")` | No |
| `os.path.exists(p)` | `True` if a file **or** folder exists there | Yes |
| `os.path.isfile(p)` | `True` if it's an existing **file** | Yes |
| `os.path.isdir(p)` | `True` if it's an existing **folder** | Yes |
| `os.path.getsize(p)` | file size in bytes | Yes |

In [ ]:
path = os.path.join("reports", "2026", "sales_report.csv")

print(os.path.basename(path))     # 'sales_report.csv'
print(os.path.dirname(path))      # 'reports/2026'
print(os.path.split(path))        # ('reports/2026', 'sales_report.csv')

name, ext = os.path.splitext(os.path.basename(path))
print(name, ext)                  # 'sales_report' '.csv'

The disk-checking functions need real paths. `.` (the cwd) always exists, and if the notebook was started from its own folder, so does this notebook file:

In [ ]:
print(os.path.exists("."))                        # True
print(os.path.isdir("."))                         # True: it's a folder
print(os.path.isfile("."))                        # False: not a file

print(os.path.exists("02_os_and_paths.ipynb"))    # this notebook
print(os.path.isfile("02_os_and_paths.ipynb"))
print(os.path.exists("does_not_exist.txt"))       # False

---
## 6. Listing a folder: `os.listdir()`

**What it is:** lists what's inside a folder, like `ls`.

**Behind the scenes:** the operating system reads the folder's entries and Python returns their **names**. It isn't recursive (it doesn't look inside sub-folders) and the order is not guaranteed.

**Syntax:** `os.listdir(path=".")`
**Returns:** a list of name strings, files and folders mixed together. **Names only, not full paths.**

In [ ]:
items = os.listdir(".")
print(items)
print(len(items), "items")

Because `listdir` gives you names only, join each name with its folder to get a path you can check or open. (`sorted()` returns a sorted copy of a list; it's covered in notebook 06.)

In [ ]:
folder = "."
for name in sorted(os.listdir(folder)):
    full = os.path.join(folder, name)          # name -> usable path
    if os.path.isdir(full):
        print("[DIR] ", name)
    else:
        print("[FILE]", name, "-", os.path.getsize(full), "bytes")

In [ ]:
# Keep only the notebooks
notebooks = []
for name in os.listdir("."):
    if name.endswith(".ipynb"):
        notebooks.append(name)
print(sorted(notebooks))

Listing a folder that doesn't exist raises `FileNotFoundError`:

In [ ]:
try:
    os.listdir("no_such_folder")
except FileNotFoundError as e:
    print("FileNotFoundError:", e)

---
## 7. Creating, renaming and removing

| Function | What it does | Fails with |
|---|---|---|
| `os.mkdir(p)` | creates **one** folder | `FileExistsError` if it exists, `FileNotFoundError` if the parent is missing |
| `os.makedirs(p, exist_ok=True)` | creates a folder **and any missing parents** | nothing, thanks to `exist_ok=True` |
| `os.rename(src, dst)` | renames or moves a file or folder | `FileNotFoundError` if `src` is missing |
| `os.remove(p)` | deletes a **file** | `FileNotFoundError` if missing |
| `os.rmdir(p)` | deletes an **empty** folder | `OSError` if the folder isn't empty |

**Careful:** `os.remove()` and `os.rmdir()` delete permanently. There's no recycle bin.

Everything below happens inside a `practice_files` folder (git-ignored), so nothing else is touched.

In [ ]:
nested = os.path.join("practice_files", "os_demo", "2026", "october")

os.makedirs(nested, exist_ok=True)      # creates all four levels in one go
print(os.path.isdir(nested))

os.makedirs(nested, exist_ok=True)      # running it again is safe
print("still fine")

In [ ]:
try:
    os.mkdir(nested)                    # mkdir refuses if the folder already exists
except FileExistsError as e:
    print("FileExistsError:", e)

Create a small file to work with (writing files is explained in [03_file_handling.ipynb](03_file_handling.ipynb)), then rename it:

In [ ]:
draft_path = os.path.join(nested, "draft.txt")
with open(draft_path, "w", encoding="utf-8") as f:
    f.write("hello")

final_path = os.path.join(nested, "final.txt")
os.rename(draft_path, final_path)       # draft.txt -> final.txt
print(os.listdir(nested))

In [ ]:
try:
    os.rmdir(nested)                    # not empty: final.txt is inside
except OSError as e:
    print(type(e).__name__ + ":", e)

In [ ]:
os.remove(final_path)                   # delete the file first...
os.rmdir(nested)                        # ...then the now-empty folder
print(os.path.exists(nested))           # False

---
## 8. Paths in `.py` scripts: relative to the cwd, not to the file

This catches out almost everyone. In a script, a relative path is resolved against the **cwd you ran `python` from**, not the folder the script is saved in.

Say `Assignment/s06-python-3/report.py` does `open("data.txt")`:

| You run | cwd | Python looks for |
|---|---|---|
| `cd Assignment/s06-python-3` then `python report.py` | `.../s06-python-3` | `.../s06-python-3/data.txt` (found) |
| `python Assignment/s06-python-3/report.py` from the repo root | the repo root | `<repo root>/data.txt` (not found!) |

**The fix:** build paths from the script's own location. Python sets the variable `__file__` in every script to that script's path:

```python
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))   # the folder this .py file is in
DATA_PATH = os.path.join(SCRIPT_DIR, "data.txt")           # works from any cwd
```

`__file__` doesn't exist in notebooks (no `.py` file is being run), so this pattern is for scripts. [solution.py](solution.py) in this folder uses it.

**See it yourself.** [paths_demo.py](paths_demo.py) prints both kinds of path. Run it from the repo root and then from inside this folder, and compare:

```bash
python Assignment/s06-python-3/paths_demo.py
cd Assignment/s06-python-3 && python paths_demo.py
```

### A note on `pathlib`

Modern Python code often uses `pathlib.Path` instead of `os.path`, for example `Path("data") / "notes.txt"`. The ideas are the same: cwd, relative vs absolute, joining parts. You'll see both styles in real projects.

---
## 9. Practice

**1.** Print the cwd, then print the absolute path of its parent folder using `os.path.abspath("..")`.

In [ ]:
# Exercise 1

**2.** Using `os.path.join`, build the path `practice_files/exercise/notes.txt`. Print its basename, its dirname and its extension.

In [ ]:
# Exercise 2

**3.** Create the folder `practice_files/exercise` with `os.makedirs` so that it's safe to run twice, then confirm it exists with `os.path.isdir`.

In [ ]:
# Exercise 3

**4.** Write a function `list_files(folder, ext)` that returns the names of the **files** in `folder` that end with `ext`. If the folder doesn't exist, catch the error and return an empty list. Test it with `"."` and `".ipynb"`, and with a folder that doesn't exist.

In [ ]:
# Exercise 4